# Astra v2 — word-level GPU training (distinct filenames)

Trains the 8.1M word-level model on a free T4:
1. **word-prose** 5400 steps → downloads as **`astra_word_prose_final.npz`**
2. **word-chat** 3800 steps → downloads as **`astra_word_chat_final.npz`**

Every checkpoint download has a **unique name**, so nothing in your Downloads folder can be confused with something else.

Runtime: **Settings/Runtime → GPU (T4)**, Internet ON. Run cells top to bottom.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

**Diagnostic — what exists on this VM** (run this whenever you're unsure):

In [ ]:
import os
paths = [
    '/content/astra',
    '/content/astra/tokenizer/artifacts/prose_chat_word.json',
    '/content/runs/word_prose/final.npz',
    '/content/runs/word_chat/resumed/final.npz',
    '/content/drive/MyDrive/astra_checkpoints/astra_word_prose_final.npz',
    '/content/drive/MyDrive/astra_checkpoints/astra_word_chat_final.npz',
]
for p in paths:
    ok = os.path.exists(p)
    sz = os.path.getsize(p) if ok else 0
    print(f"{'YES' if ok else 'NO '}  {sz:>12,}  {p}")

In [ ]:
import os, subprocess
REPO = '/content/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed - check Internet is ON and retry'
os.chdir(REPO)
print('repo ready at', os.getcwd())

**Mount Google Drive** — persistent backup in case the VM recycles:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/astra_checkpoints', exist_ok=True)
print('Drive ready')

**Build the word tokenizer artifact** (gitignored in the repo):

In [ ]:
import os
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    !python tools/build_word_tokenizer.py
import json
a = json.load(open('tokenizer/artifacts/prose_chat_word.json'))
print('tokenizer vocab:', a['vocab_size'])

---
## STAGE 1 — word-prose base (5400 steps, ~5 min)
Ends with **`[step 5400]`** and auto-downloads as **`astra_word_prose_final.npz`**.

In [ ]:
import os, subprocess, shutil
if not os.path.isdir('/content/astra'):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git /content/astra', shell=True, check=True)
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
!mkdir -p /content/runs
!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 5400 --out /content/runs/word_prose \
  --cache-dir /content/cache 2>&1 | tail -10
SRC = '/content/runs/word_prose/final.npz'
assert os.path.exists(SRC), 'PROSE NOT PRODUCED - training failed'
# copy to a unique name, then download
shutil.copy(SRC, '/content/astra_word_prose_final.npz')
shutil.copy(SRC, '/content/drive/MyDrive/astra_checkpoints/astra_word_prose_final.npz')
print('PROSE ok:', os.path.getsize('/content/astra_word_prose_final.npz'), 'bytes')
from google.colab import files
files.download('/content/astra_word_prose_final.npz')

---
## STAGE 2 — word-chat fine-tune (3800 steps, ~15 min)
**Only run this after STAGE 1 finished.** Warm-starts from the prose final.
Ends with **`[step 3800]`** and auto-downloads as **`astra_word_chat_final.npz`**.

In [ ]:
import os, subprocess, shutil
if not os.path.isdir('/content/astra'):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git /content/astra', shell=True, check=True)
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
PROSE = '/content/runs/word_prose/final.npz'
assert os.path.exists(PROSE), 'NO PROSE FINAL - run STAGE 1 first'
!python training/gpu_train.py --config configs/astra5m_word_chat.json \
  --steps 3800 --out /content/runs/word_chat \
  --resume {PROSE} --reset-step \
  --cache-dir /content/cache 2>&1 | tail -10
SRC = '/content/runs/word_chat/resumed/final.npz'
assert os.path.exists(SRC), 'CHAT FINAL NOT PRODUCED - training failed'
# copy to a unique name, then download
shutil.copy(SRC, '/content/astra_word_chat_final.npz')
shutil.copy(SRC, '/content/drive/MyDrive/astra_checkpoints/astra_word_chat_final.npz')
print('CHAT ok:', os.path.getsize('/content/astra_word_chat_final.npz'), 'bytes')
from google.colab import files
files.download('/content/astra_word_chat_final.npz')

---
## Download / re-download (optional)
Pulls from the VM **and** Drive with the distinct names. Only files that exist are downloaded.

In [ ]:
from google.colab import files
import os, shutil
cands = {
    'astra_word_prose_final.npz': [
        '/content/runs/word_prose/final.npz',
        '/content/drive/MyDrive/astra_checkpoints/astra_word_prose_final.npz',
    ],
    'astra_word_chat_final.npz': [
        '/content/runs/word_chat/resumed/final.npz',
        '/content/drive/MyDrive/astra_checkpoints/astra_word_chat_final.npz',
    ],
}
for name, paths in cands.items():
    src = next((p for p in paths if os.path.exists(p)), None)
    if src is None:
        print(f'NO FILE: {name} (neither VM nor Drive has it)')
        continue
    dst = f'/content/{name}'
    shutil.copy(src, dst)
    print(f'DOWNLOADING {name} ({os.path.getsize(dst):,} bytes) from {src}')
    files.download(dst)

In [ ]:
import hashlib
for name in ['astra_word_prose_final.npz', 'astra_word_chat_final.npz']:
    p = f'/content/{name}'
    if os.path.exists(p):
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()[:12]
        print(name, os.path.getsize(p), 'md5', h)
    else:
        print(name, 'NOT PRESENT')